# ESP-Lab workflow figure gallery

Discover the figures that actually exist in the workflow output directory and build a searchable, responsive webpage. Only supported image files matching `fig_*` are included. Existing titles and captions are retained, stale manifest entries are removed, and metadata is generated for newly discovered figures.

In [ ]:
# ============================================================
# Configuration
# ============================================================
from pathlib import Path

DIAG_DIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_PATTERN = "fig_*"
PORTAL_URL = "https://portal.nersc.gov/cfs/e3sm/zhan391/esp-lab_diag/index.html"

In [ ]:
# Imports
import sys

import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "esp_lab").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from esp_lab.diagnostics.web import (
    discover_workflow_figures,
    generate_diagnostics_webpage,
)

## 1. Discover workflow figures

Scan the directory rather than trusting old manifest entries. The synchronized catalog below is exactly what the webpage will show.

In [ ]:
manifest = discover_workflow_figures(
    DIAG_DIR,
    pattern=FIGURE_PATTERN,
    write_manifest=True,
)
catalog = pd.DataFrame(manifest["figures"])[
    ["file", "group", "mode", "metric", "title", "caption"]
]
print(f"Found {len(catalog)} workflow figures in {DIAG_DIR}")
display(catalog.groupby("group").size().rename("figures").to_frame())
display(catalog)

## 2. Build the webpage

In [ ]:
output_html = generate_diagnostics_webpage(
    DIAG_DIR,
    discover_figures=True,
    figure_pattern=FIGURE_PATTERN,
)
print(f"Webpage created with {len(catalog)} figures: {output_html}")
display(HTML(f'<a href="{PORTAL_URL}" target="_blank">Open the ESP-Lab figure gallery</a>'))

The resulting `index.html` provides group and metric filters, text search, responsive figure cards, full-resolution viewing, and figure comparison. Generation also makes the cataloged figures publicly readable and the gallery directory publicly traversable so the portal can serve every image. Re-run the two sections whenever workflow figures are added, replaced, or removed.